# Обучение детектора AI-текста

Ноутбук повторяет основной пайплайн проекта: загрузка объединённого датасета, разбиение на train/test, обучение через PyTorch Lightning, оценка на отложенной выборке, разбор по источникам и примеры инференса.

Все основные настройки находятся прямо в ноутбуке, а архитектура модели и Lightning-обвязка определены внутри этого `ipynb`.

In [ ]:
from __future__ import annotations

import json
import math
import os
import warnings
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from sklearn.metrics import f1_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader

from data import BPETokenizer, TextClassificationDataset, collate_fn
from dataset_loader import DatasetConfig, Sample, load_combined_dataset
from evaluate import evaluate_model, evaluate_per_source


In [ ]:
warnings.filterwarnings(
    "ignore",
    module=r"pytorch_lightning\.utilities\._pytree",
)
warnings.filterwarnings(
    "ignore",
    message=r".*LeafSpec.*deprecated.*",
)
warnings.filterwarnings(
    "ignore",
    message=r".*lr_scheduler\.step\(\).*before `optimizer\.step\(\)`.*",
)


class PositionalEncoding(nn.Module):
    """Синусоидальное позиционное кодирование."""

    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


class AITextDetector(nn.Module):
    """Transformer Encoder для бинарной классификации текста."""

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 512,
        max_len: int = 512,
        dropout: float = 0.1,
        pad_idx: int = 0,
        unk_idx: int = 1,
        token_dropout: float = 0.05,
    ):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.unk_idx = unk_idx
        self.token_dropout = token_dropout

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(d_model, max_len + 1, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        try:
            self.transformer = nn.TransformerEncoder(
                encoder_layer,
                num_layers=num_layers,
                enable_nested_tensor=False,
            )
        except TypeError:
            self.transformer = nn.TransformerEncoder(
                encoder_layer,
                num_layers=num_layers,
            )
        self.layer_norm = nn.LayerNorm(d_model)

        pooled_dim = d_model * 3
        self.head = nn.Sequential(
            nn.Linear(pooled_dim, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for parameter in self.parameters():
            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)

    def _make_padding_mask(self, input_ids: torch.Tensor) -> torch.Tensor:
        cls_mask = torch.zeros(input_ids.size(0), 1, dtype=torch.bool, device=input_ids.device)
        pad_mask = input_ids == self.pad_idx
        return torch.cat([cls_mask, pad_mask], dim=1)

    def _apply_token_dropout(self, input_ids: torch.Tensor) -> torch.Tensor:
        if not self.training or self.token_dropout <= 0:
            return input_ids

        drop_mask = torch.rand_like(input_ids, dtype=torch.float32) < self.token_dropout
        drop_mask &= input_ids != self.pad_idx
        if not torch.any(drop_mask):
            return input_ids

        dropped = input_ids.clone()
        dropped[drop_mask] = self.unk_idx
        return dropped

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        batch_size = input_ids.size(0)
        input_ids = self._apply_token_dropout(input_ids)
        x = self.embedding(input_ids) * math.sqrt(self.d_model)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = self.pos_encoder(x)

        padding_mask = self._make_padding_mask(input_ids)
        x = self.transformer(x, src_key_padding_mask=padding_mask)
        x = self.layer_norm(x)

        cls_repr = x[:, 0]
        token_repr = x[:, 1:]
        token_mask = ~padding_mask[:, 1:]
        token_mask_f = token_mask.unsqueeze(-1).to(token_repr.dtype)

        denom = token_mask_f.sum(dim=1).clamp_min(1.0)
        mean_repr = (token_repr * token_mask_f).sum(dim=1) / denom

        masked_tokens = token_repr.masked_fill(~token_mask.unsqueeze(-1), float("-inf"))
        max_repr = masked_tokens.max(dim=1).values
        max_repr = torch.where(torch.isfinite(max_repr), max_repr, torch.zeros_like(max_repr))

        pooled = torch.cat([cls_repr, mean_repr, max_repr], dim=-1)
        logits = self.head(pooled).squeeze(-1)
        return logits


@dataclass
class TrainConfig:
    d_model: int = 256
    nhead: int = 8
    num_layers: int = 4
    dim_feedforward: int = 512
    max_len: int = 512
    dropout: float = 0.1
    epochs: int = 10
    batch_size: int = 32
    lr: float = 3e-4
    weight_decay: float = 0.01
    warmup_steps: int = 100
    max_vocab: int = 30000
    val_split: float = 0.1
    seed: int = 42
    use_amp: bool = True
    token_dropout: float = 0.05
    label_smoothing: float = 0.02
    patience: int = 3
    grad_clip_val: float = 1.0
    accumulate_grad_batches: int = 1
    num_workers: int = -1
    save_dir: str = "checkpoints/notebook_run"


@dataclass
class DetectionResult:
    label: str
    confidence: float
    logit: float


class NotebookDetector:
    """Упрощённая обёртка для инференса внутри ноутбука."""

    def __init__(
        self,
        model: AITextDetector,
        tokenizer: BPETokenizer,
        device: str = "auto",
        threshold: float = 0.5,
        use_amp: bool = True,
    ):
        self.device = torch.device(
            device if device != "auto" else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.model = model.to(self.device).eval()
        self.tokenizer = tokenizer
        self.threshold = threshold
        self.use_amp = use_amp

    @torch.no_grad()
    def predict(self, text: str) -> DetectionResult:
        ids = torch.tensor([self.tokenizer.encode(text)], dtype=torch.long, device=self.device)
        with torch.autocast(
            device_type=self.device.type,
            enabled=self.use_amp and self.device.type != "cpu",
        ):
            logit = self.model(ids).item()
        prob = torch.sigmoid(torch.tensor(logit)).item()
        is_ai = prob >= self.threshold
        return DetectionResult(
            label="AI" if is_ai else "Human",
            confidence=prob if is_ai else 1 - prob,
            logit=logit,
        )


class SmoothedBCEWithLogitsLoss(nn.Module):
    def __init__(self, smoothing: float = 0.0):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if self.smoothing > 0:
            targets = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(logits, targets)


def compute_optimal_threshold(logits: np.ndarray, labels: np.ndarray) -> tuple[float, float]:
    probs = 1 / (1 + np.exp(-logits))
    precision, recall, thresholds = precision_recall_curve(labels, probs)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
    best_idx = int(np.argmax(f1_scores))
    threshold = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.5
    preds = (probs >= threshold).astype(int)
    return threshold, float(f1_score(labels, preds, average="macro"))


def build_stratify_labels(labels: list[int]) -> list[int] | None:
    if len(set(labels)) < 2:
        return None
    class_counts = {label: labels.count(label) for label in set(labels)}
    if min(class_counts.values()) < 2:
        return None
    return labels


def split_train_val(
    texts: list[str],
    labels: list[int],
    val_split: float,
    seed: int,
) -> tuple[list[str], list[str], list[int], list[int]]:
    if len(texts) != len(labels):
        raise ValueError("texts и labels должны иметь одинаковую длину")

    if not 0 < val_split < 1 or len(texts) < 2:
        return texts, [], labels, []

    stratify = build_stratify_labels(labels)
    try:
        train_texts, val_texts, train_labels, val_labels = train_test_split(
            texts,
            labels,
            test_size=val_split,
            random_state=seed,
            stratify=stratify,
        )
    except ValueError:
        train_texts, val_texts, train_labels, val_labels = train_test_split(
            texts,
            labels,
            test_size=val_split,
            random_state=seed,
            stratify=None,
        )
    return train_texts, val_texts, train_labels, val_labels


def split_loaded_data(result, test_ratio: float, seed: int):
    indices = list(range(len(result.texts)))
    stratify = None

    source_label_groups = [
        f"{result.samples[i].source_dataset}:{result.labels[i]}"
        for i in indices
    ]
    group_counts = Counter(source_label_groups)
    if group_counts and min(group_counts.values()) >= 2:
        stratify = source_label_groups
    else:
        label_counts = Counter(result.labels)
        if label_counts and min(label_counts.values()) >= 2:
            stratify = result.labels

    try:
        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_ratio,
            random_state=seed,
            stratify=stratify,
        )
    except ValueError:
        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_ratio,
            random_state=seed,
            stratify=None,
        )

    train_texts = [result.texts[i] for i in train_idx]
    train_labels = [result.labels[i] for i in train_idx]
    test_texts = [result.texts[i] for i in test_idx]
    test_labels = [result.labels[i] for i in test_idx]
    test_samples = [result.samples[i] for i in test_idx]
    return train_texts, train_labels, test_texts, test_labels, test_samples


def calibrate_threshold(model: AITextDetector, loader: DataLoader, use_amp: bool) -> float:
    if len(loader.dataset) == 0:
        return 0.5

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()
    logits_list: list[float] = []
    labels_list: list[int] = []

    with torch.no_grad():
        for input_ids, targets in loader:
            input_ids = input_ids.to(device)
            with torch.autocast(
                device_type=device.type,
                enabled=use_amp and device.type != "cpu",
            ):
                logits = model(input_ids)
            logits_list.extend(logits.cpu().tolist())
            labels_list.extend(targets.int().cpu().tolist())

    logits_arr = np.array(logits_list, dtype=np.float32)
    labels_arr = np.array(labels_list, dtype=np.int64)
    threshold, _ = compute_optimal_threshold(logits_arr, labels_arr)
    return threshold


def configure_torch_runtime() -> None:
    if not torch.cuda.is_available():
        return
    torch.set_float32_matmul_precision("high")
    if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends.cudnn, "allow_tf32"):
        torch.backends.cudnn.allow_tf32 = True


def resolve_num_workers(num_workers: int) -> int:
    if num_workers >= 0:
        return num_workers
    cpu_count = os.cpu_count() or 2
    return max(1, min(11, cpu_count - 1))


class AIDetectorModule(pl.LightningModule):
    """LightningModule для обучения детектора."""

    def __init__(self, model: AITextDetector, cfg: TrainConfig, total_steps: int):
        super().__init__()
        self.model = model
        self.cfg = cfg
        self.total_steps = total_steps
        self.criterion = SmoothedBCEWithLogitsLoss(cfg.label_smoothing)
        self.validation_logits: list[torch.Tensor] = []
        self.validation_targets: list[torch.Tensor] = []

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        return self.model(input_ids)

    def training_step(self, batch, batch_idx):
        input_ids, targets = batch
        logits = self(input_ids)
        loss = self.criterion(logits, targets)
        acc = ((logits > 0).long() == targets.long()).float().mean()
        self.log("train_loss", loss, on_epoch=True, on_step=False, prog_bar=True)
        self.log("train_acc", acc, on_epoch=True, on_step=False, prog_bar=True)
        return loss

    def on_validation_epoch_start(self):
        self.validation_logits.clear()
        self.validation_targets.clear()

    def validation_step(self, batch, batch_idx):
        input_ids, targets = batch
        logits = self(input_ids)
        loss = self.criterion(logits, targets)
        self.validation_logits.append(logits.detach().cpu())
        self.validation_targets.append(targets.detach().cpu())
        self.log("val_loss", loss, on_epoch=True, prog_bar=True)

    def on_validation_epoch_end(self):
        if not self.validation_logits:
            return

        logits = torch.cat(self.validation_logits).numpy()
        labels = torch.cat(self.validation_targets).numpy().astype(int)
        probs = 1 / (1 + np.exp(-logits))
        preds = (probs >= 0.5).astype(int)
        threshold, best_f1 = compute_optimal_threshold(logits, labels)

        self.log("val_acc", float((preds == labels).mean()), prog_bar=True)
        self.log("val_f1_macro", f1_score(labels, preds, average="macro"), prog_bar=True)
        self.log("val_best_f1", best_f1, prog_bar=False)
        self.log("val_best_threshold", threshold, prog_bar=False)
        if len(np.unique(labels)) > 1:
            self.log("val_roc_auc", roc_auc_score(labels, probs), prog_bar=False)

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.cfg.lr, weight_decay=self.cfg.weight_decay)
        warmup_steps = min(self.cfg.warmup_steps, max(self.total_steps - 1, 0))

        def lr_lambda(current_step: int) -> float:
            if self.total_steps <= 1:
                return 1.0
            if warmup_steps > 0 and current_step < warmup_steps:
                return 0.1 + 0.9 * (current_step / max(1, warmup_steps))

            progress = (current_step - warmup_steps) / max(1, self.total_steps - warmup_steps)
            progress = min(max(progress, 0.0), 1.0)
            return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

        scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }


def save_test_split(test_samples: list[Sample], output_path: str | Path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as handle:
        for sample in test_samples:
            handle.write(
                json.dumps(
                    {
                        "text": sample.text,
                        "label": int(sample.label),
                        "source_dataset": sample.source_dataset,
                        "generator": sample.generator,
                        "domain": sample.domain,
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )


def train_pipeline(
    texts: list[str],
    labels: list[int],
    cfg: TrainConfig,
) -> tuple[AITextDetector, BPETokenizer, float]:
    torch.manual_seed(cfg.seed)
    configure_torch_runtime()
    num_workers = resolve_num_workers(cfg.num_workers)

    train_texts, val_texts, train_labels, val_labels = split_train_val(
        texts,
        labels,
        cfg.val_split,
        cfg.seed,
    )
    tokenizer = BPETokenizer.from_texts(
        train_texts,
        max_vocab=cfg.max_vocab,
        max_len=cfg.max_len,
    )
    train_ds = TextClassificationDataset(train_texts, train_labels, tokenizer)
    val_ds = TextClassificationDataset(val_texts, val_labels, tokenizer)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=num_workers,
        persistent_workers=num_workers > 0,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        collate_fn=collate_fn,
        num_workers=num_workers,
        persistent_workers=num_workers > 0,
        pin_memory=torch.cuda.is_available(),
    )

    print(f"Словарь: {tokenizer.vocab_size} | Train: {len(train_ds)} | Val: {len(val_ds)}")

    model = AITextDetector(
        vocab_size=tokenizer.vocab_size,
        d_model=cfg.d_model,
        nhead=cfg.nhead,
        num_layers=cfg.num_layers,
        dim_feedforward=cfg.dim_feedforward,
        max_len=cfg.max_len,
        dropout=cfg.dropout,
        pad_idx=tokenizer.pad_idx,
        unk_idx=tokenizer.unk_idx,
        token_dropout=cfg.token_dropout,
    )

    n_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    print(f"Количество обучаемых параметров: {n_params:,}")

    steps_per_epoch = max(1, math.ceil(len(train_loader) / max(cfg.accumulate_grad_batches, 1)))
    total_steps = cfg.epochs * steps_per_epoch
    lit_model = AIDetectorModule(model, cfg, total_steps)

    save_dir = Path(cfg.save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    logger = CSVLogger(save_dir=str(save_dir), name="logs")
    checkpoint_dir = Path(logger.log_dir) / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_cb = ModelCheckpoint(
        dirpath=checkpoint_dir,
        filename="best_checkpoint",
        monitor="val_f1_macro",
        mode="max",
        save_top_k=1,
    )
    early_stopping_cb = EarlyStopping(
        monitor="val_f1_macro",
        mode="max",
        patience=cfg.patience,
    )

    use_amp = cfg.use_amp and torch.cuda.is_available()
    trainer = pl.Trainer(
        max_epochs=cfg.epochs,
        callbacks=[checkpoint_cb, early_stopping_cb],
        precision="16-mixed" if use_amp else "32-true",
        enable_progress_bar=True,
        log_every_n_steps=10,
        gradient_clip_val=cfg.grad_clip_val,
        accumulate_grad_batches=cfg.accumulate_grad_batches,
        logger=logger,
        enable_model_summary=False,
    )
    trainer.fit(lit_model, train_loader, val_loader)

    best_ckpt = torch.load(checkpoint_cb.best_model_path, weights_only=True)
    state_dict = {
        key.removeprefix("model."): value
        for key, value in best_ckpt["state_dict"].items()
        if key.startswith("model.")
    }
    torch.save(state_dict, save_dir / "best_model.pt")
    tokenizer.save(save_dir / "tokenizer")

    model_config = {
        "vocab_size": tokenizer.vocab_size,
        "d_model": cfg.d_model,
        "nhead": cfg.nhead,
        "num_layers": cfg.num_layers,
        "dim_feedforward": cfg.dim_feedforward,
        "max_len": cfg.max_len,
        "dropout": cfg.dropout,
        "pad_idx": tokenizer.pad_idx,
        "unk_idx": tokenizer.unk_idx,
        "token_dropout": 0.0,
    }
    (save_dir / "model_config.json").write_text(
        json.dumps(model_config, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    best_val_f1 = checkpoint_cb.best_model_score
    if best_val_f1 is not None:
        print(f"Лучший macro F1 на валидации: {float(best_val_f1):.3f}")

    model.load_state_dict(state_dict)
    calibrated_threshold = calibrate_threshold(model, val_loader, use_amp)
    (save_dir / "inference_config.json").write_text(
        json.dumps({"threshold": calibrated_threshold}, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"Откалиброванный порог: {calibrated_threshold:.3f}")
    return model, tokenizer, calibrated_threshold


def print_eval_summary(result):
    print("\nИтоговая оценка на тестовой выборке")
    print("-" * 60)
    print(f"Accuracy:      {result.accuracy:.4f}")
    print(f"Macro F1:      {result.f1:.4f}")
    print(f"ROC-AUC:       {result.roc_auc:.4f}")
    print(f"Порог:         {result.threshold_used:.3f}")
    print(f"Лучший порог:  {result.optimal_threshold:.3f}")
    print()
    print(f"Precision Human: {result.precision_human:.4f}")
    print(f"Recall Human:    {result.recall_human:.4f}")
    print(f"F1 Human:        {result.f1_human:.4f}")
    print(f"Precision AI:    {result.precision_ai:.4f}")
    print(f"Recall AI:       {result.recall_ai:.4f}")
    print(f"F1 AI:           {result.f1_ai:.4f}")
    print()
    print("Матрица ошибок:")
    print(result.confusion)
    print()
    print("Полный classification report:")
    print(result.report)


def print_source_summary(breakdowns):
    print("\nМетрики по источникам")
    print("-" * 60)
    for item in breakdowns:
        print(
            f"{item.source:20s} | сэмплов={item.n_samples:6d} | "
            f"accuracy={item.accuracy:.4f} | f1={item.f1:.4f} | roc_auc={item.roc_auc:.4f}"
        )


def label_to_russian(label: str) -> str:
    return "AI" if label == "AI" else "Человек"


In [ ]:
# Основные настройки ноутбука

ALL_SOURCES = [
    "raid",
    "ai_pile",
    "gpt_wiki",
    "human_vs_ai",
    "ai_human_mixed",
    "hc3",
    "coat",
    "ruatd",
    "daigt_proper",
    "daigt_v2",
    "m_daigt",
    "gsingh_train",
]

artifact_dir = Path("checkpoints/notebook_run")
test_ratio = 0.15

source_paths = {
    # При необходимости укажите локальные пути для источников, которые не скачиваются автоматически.
    # "daigt_proper": "/абсолютный/путь/к/train_v2_drcat_02.csv",
    # "m_daigt": "/абсолютный/путь/к/каталогу_m_daigt",
}

data_config = DatasetConfig(
    sources=ALL_SOURCES,
    source_paths=source_paths,
    max_per_source=100_000,
    max_total=200_000,
    min_text_length=35,
    max_text_length=3_000,
    balance_labels=True,
    balance_sources=False,
    seed=42,
    cache_dir=".cache/datasets",
    auto_download_kaggle=True,
    cache_sources=True,
)

train_config = TrainConfig(
    d_model=256,
    nhead=8,
    num_layers=4,
    dim_feedforward=512,
    max_len=512,
    dropout=0.1,
    epochs=10,
    batch_size=32,
    lr=3e-4,
    weight_decay=0.01,
    warmup_steps=100,
    max_vocab=50_000,
    val_split=0.1,
    seed=42,
    use_amp=True,
    token_dropout=0.05,
    label_smoothing=0.02,
    patience=3,
    grad_clip_val=1.0,
    accumulate_grad_batches=1,
    num_workers=-1,
    save_dir=str(artifact_dir),
)

live_examples = [
    ("Ну блин, опять забыл зонт и промок как собака.", "Human"),
    ("Just grabbed tacos, best decision I've made all week lol", "Human"),
    ("spent 3 hours debugging only to find a missing comma", "Human"),
    (
        "The implementation of machine learning algorithms in healthcare has demonstrated significant potential for improving diagnostic accuracy.",
        "AI",
    ),
    (
        "In conclusion, the comprehensive analysis reveals a multifaceted landscape that necessitates careful consideration.",
        "AI",
    ),
    (
        "Furthermore, it is important to note that the integration of these systems requires a holistic understanding of the underlying frameworks.",
        "AI",
    ),
]

print("Каталог артефактов:", artifact_dir)
print("Источники по умолчанию:", data_config.sources)
print("Эпохи:", train_config.epochs)
print("Размер батча:", train_config.batch_size)


## Шаг 1. Загрузка объединённого датасета

In [ ]:
result = load_combined_dataset(data_config)
train_texts, train_labels, test_texts, test_labels, test_samples = split_loaded_data(
    result,
    test_ratio=test_ratio,
    seed=train_config.seed,
)

print(f"Всего примеров: {len(result.texts):,}")
print(f"Примеров для обучения: {len(train_texts):,}")
print(f"Примеров для теста: {len(test_texts):,}")
print("Распределение меток в train:", Counter(train_labels))
print("Распределение меток в test:", Counter(test_labels))
print("Источники в test:", Counter(sample.source_dataset for sample in test_samples))


## Шаг 2. Обучение через PyTorch Lightning

In [ ]:
model, tokenizer, threshold = train_pipeline(train_texts, train_labels, train_config)

artifact_dir.mkdir(parents=True, exist_ok=True)
save_test_split(test_samples, artifact_dir / "test_samples.jsonl")

run_config = {
    "artifact_dir": str(artifact_dir),
    "test_ratio": test_ratio,
    "data_config": asdict(data_config),
    "train_config": asdict(train_config),
}
(artifact_dir / "notebook_run_config.json").write_text(
    json.dumps(run_config, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Чекпоинт сохранён в: {artifact_dir}")
print(f"Тестовый сплит сохранён в: {artifact_dir / 'test_samples.jsonl'}")
print(f"Конфиг запуска сохранён в: {artifact_dir / 'notebook_run_config.json'}")
print(f"Финальный порог: {threshold:.3f}")


## Шаг 3. Оценка на отложенной тестовой выборке

In [ ]:
eval_result = evaluate_model(
    model,
    tokenizer,
    test_texts,
    test_labels,
    batch_size=train_config.batch_size,
    threshold=threshold,
)
print_eval_summary(eval_result)


## Шаг 4. Метрики по отдельным источникам

In [ ]:
detector = NotebookDetector(model, tokenizer, threshold=threshold, use_amp=train_config.use_amp)
breakdowns = evaluate_per_source(detector, test_samples)
if breakdowns:
    print_source_summary(breakdowns)
else:
    print("Для текущего тестового сплита разбор по источникам недоступен.")


## Шаг 5. Примеры инференса

In [ ]:
correct = 0
for text, expected in live_examples:
    prediction = detector.predict(text)
    ok = prediction.label == expected
    correct += int(ok)
    print(
        {
            "совпадение": ok,
            "ожидалось": label_to_russian(expected),
            "предсказано": label_to_russian(prediction.label),
            "уверенность": round(prediction.confidence, 4),
            "текст": text,
        }
    )

print(f"Точность на примерах: {correct}/{len(live_examples)}")
print(f"Каталог чекпоинта: {artifact_dir}")
print(f"Используемый порог: {threshold:.3f}")
